# Deep Agents: Building a Research Agent (Workshop Edition)

<img src="./images/deepAgentsDiag.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Deep Agents harness overview">

In the next ~30 minutes, you'll progressively build a research agent with [LangChain Deep Agents](https://docs.langchain.com/oss/python/deepagents/).

**What you'll learn:** the harness, custom tools, backends, subagents, human-in-the-loop, long-term memory, and AGENTS.md + Skills.

> Requires a [Tavily API key](https://tavily.com) for web search and an OpenAI key for the default model (`gpt-5.4`).


## Part 0: Setup

If you haven't already: install dependencies and copy `.env.example` → `.env`. See the README.

The model is configured in `utils/models.py` — defaults to `openai:gpt-5.4` for the main agent. The research subagent in Part 4 uses `openai:gpt-5.4-mini`. Edit `utils/models.py` to swap providers.


In [1]:
# Quick install if needed:
# !pip install -e .

import sys
import warnings
from pathlib import Path

from rich import print as rprint

warnings.filterwarnings("ignore")

project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
rprint(model)

from dotenv import load_dotenv
load_dotenv(override=True)

ChatOpenAI(
    output_version=None,
    profile={
        'name': 'GPT-5.4',
        'release_date': '2026-03-05',
        'last_updated': '2026-03-05',
        'open_weights': False,
        'max_input_tokens': 1050000,
        'max_output_tokens': 128000,
        'text_inputs': True,
        'image_inputs': True,
        'audio_inputs': False,
        'pdf_inputs': True,
        'video_inputs': False,
        'text_outputs': True,
        'image_outputs': False,
        'audio_outputs': False,
        'video_outputs': False,
        'reasoning_output': True,
        'tool_calling': True,
        'structured_output': True,
        'attachment': True,
        'temperature': False,
        'image_url_inputs': True,
        'pdf_tool_message': True,
        'image_tool_message': True,
        'tool_choice': True
    },
    client=<openai.resources.chat.completions.completions.Completions object at 0x10f53d6a0>,
    async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10f53e120>,
    root_client=<openai.OpenAI object at 0x10eb9aa50>,
    root_async_client=<openai.AsyncOpenAI object at 0x10f53de80>,
    model_name='gpt-5.4',
    model_kwargs={},
    openai_api_key=SecretStr('**********'),
    openai_proxy=None,
    stream_usage=True,
    stream_chunk_timeout=120.0
)

True

## Part 1: Your First Deep Agent (The Harness)

<img src="./images/deepAgentsHarnessOverview.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Deep Agents harness overview">

Deep Agents is an **agent harness** — a tool-calling loop with pre-built tools and capabilities baked in.

**You get out of the box:**
- **Filesystem tools** — `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`
- **Planning tool** — `write_todos` for task tracking
- **Subagent delegation** — `task()` tool for isolated work
- **Context management** — auto-evicts large tool results (>20k tokens) to the filesystem and summarizes history at ~85% context capacity

These come from a stack of built-in middleware (TodoList, Filesystem, SubAgent, Summarization, PatchToolCalls) automatically attached when you call `create_deep_agent()`. You can also pass your own via `middleware=[...]`.



In [2]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model
)

# Even with no custom tools, the agent can write/read files
result = agent.invoke({
    "messages": [{
        "role": "user", 
        "content": "Write a limerick about the World Cup to a file called limerick.md"}]
})

print("Agent Chat Response:", result["messages"][-1].content)
print("\nVirtual filesystem:", list(result.get("files", {}).keys()))
print("\nLimerick.md Contents:")
rprint(result["files"]["/limerick.md"]["content"])

/Users/justin/dev/ls-deepagents/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Agent Chat Response: Created `/limerick.md`.

Virtual filesystem: ['/limerick.md']

Limerick.md Contents:


There once was a grand World Cup show,
Where nations made dazzling displays glow.
With a kick and a cheer,
Fans roared far and near,
As dreams danced in each dazzling goal.

**Key takeaway:** `create_deep_agent()` gives you filesystem + planning capabilities, and files default to ephemeral agent state — we'll customize that in Part 3.

## Part 2: Adding Custom Tools

A research agent needs **web search**. Let's add a Tavily tool with the `@tool` decorator.


In [3]:
from langchain.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool(parse_docstring=True)
def tavily_search(query: str) -> str:
    """Search the web for information on a given query.

    Args:
        query: Search query to execute
    """
    results = tavily_client.search(query, max_results=3, topic="general")
    chunks = [
        f"## {r['title']}\n**URL:** {r['url']}\n\n{r.get('content', '')}\n\n---\n"
        for r in results.get("results", [])
    ]
    return f"Found {len(chunks)} result(s) for '{query}':\n\n{''.join(chunks)}"

agent = create_deep_agent(
    model=model,
    tools=[tavily_search]  # <-- Here is our new search tool
)

# Quick test - keep prompts succinct to keep demos fast
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Give me a summary of Morocco's 2026 World Cup wins/losses thus far."
            "Find their next opponent and give me thier prospects for winning."
        ),
    }]
})
rprint(result["messages"][-1].content)


Morocco are unbeaten so far at the 2026 World Cup.

Record so far
- Draw vs Brazil: 1–1
- Win vs Scotland: 1–0
- Win vs Haiti: 4–2
- Win vs Canada: 3–0 in the Round of 16

Overall
- 3 wins, 1 draw, 0 losses
- Goals scored: 9
- Goals conceded: 3

Next opponent
- France in the quarterfinals
- Scheduled for July 9, 2026, in Foxborough/Gillette Stadium

Prospects for winning
- Morocco definitely has a real chance. They’ve been one of the tournament’s strongest defensive and transition 
teams, and beating Canada 3–0 after getting through a tough group is a strong sign.
- The best evidence in their favor is that they’re still unbeaten, already drew Brazil, and have only conceded 3 
goals in 4 matches.
- That said, France should be considered the favorite. They have more top-end depth, more attacking firepower, and 
more experience in late-round knockout matches.

Bottom line
- Morocco are credible underdogs, not long shots.
- A Morocco win would be an upset, but not a shocking one.
- Most neutral previews would likely put this around:
  - France favored
  - Morocco with a solid chance if they keep it tight and create on the counter

If you want, I can also give you a short tactical preview of Morocco vs. France.

**Key takeaway:** the `@tool` decorator turns any Python function into a LangChain tool — pass tools via `tools=[...]`.


## Part 3: Understanding Backends

<img src="./images/deepAgentBackends.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Backend Architecture">

Where do the agent's files actually go? **Backends** are pluggable storage. The question that actually matters when choosing one: **who can see the files?**

| Backend | Who can see them | Use case |
|---|---|---|
| **StateBackend** *(default)* | This thread only | Scratch pad, intermediate results, large tool-output eviction |
| **FilesystemBackend** | Anything with access to that directory | Local projects, CI sandboxes, mounted volumes |
| **StoreBackend** | Any thread in the same namespace | Long-term memories (Part 6) |
| **CompositeBackend** | Depends on the route | Mix backends (Part 6) |

By default, `create_deep_agent()` uses `StateBackend` — files are stored in agent state and disappear when the thread ends.


In [4]:
from langgraph.checkpoint.memory import MemorySaver
from langsmith import uuid7

# Checkpointer lets us persist within a thread; thread_id selects the conversation
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are a helpful research assistant.",
    checkpointer=checkpointer,
)

# Thread 1: write a file
config1 = {"configurable": {"thread_id": str(uuid7())}}
tid1 = config1["configurable"]["thread_id"]
print(f"Thread 1: ...{tid1[-8:]}")
agent.invoke({
    "messages": [
        {"role": "user", 
         "content": "Write a file called /world_cup_predictions.md "
         "with a ranked list of how you predict the 2026 World Cup teams will finish," 
         " only include the top 10."}]
}, config=config1)

# Now look the file back up BY THREAD ID — no new invoke, straight from the checkpointer.
# We only need the thread_id; get_state deserializes the last saved snapshot for it.
thread_1_snapshot = agent.get_state({"configurable": {"thread_id": tid1}})
files = thread_1_snapshot.values.get("files", {})
print("Thread 1 files (looked up by id):", list(files.keys()))
print("Thread 1 /world_cup_predictions.md:", files["/world_cup_predictions.md"]["content"])

# Thread 2: brand-new thread → look it up by ITS id and the file is gone (StateBackend is ephemeral!)
config2 = {"configurable": {"thread_id": str(uuid7())}}
tid2 = config2["configurable"]["thread_id"]
agent.invoke({
    "messages": [{"role": "user", "content": "List all files with ls /"}]
}, config=config2)

thread_2_snapshot = agent.get_state({"configurable": {"thread_id": tid2}})
print(f"\nThread 2: ...{tid2[-8:]}")
print("Thread 2 files (looked up by id):", list(thread_2_snapshot.values.get("files", {}).keys()))

Thread 1: ...b5123543
Thread 1 files (looked up by id): ['/world_cup_predictions.md']
Thread 1 /world_cup_predictions.md: # 2026 FIFA World Cup Predictions

1. Brazil
2. France
3. Argentina
4. England
5. Spain
6. Portugal
7. Germany
8. Netherlands
9. Uruguay
10. Italy


Thread 2: ...fdcd33a5
Thread 2 files (looked up by id): []


**Key takeaway**: the choice between backends is about who can see the files - just this thread (StateBackend), anyone with disk access (FilesystemBackend), or other threads in the same namespace (StoreBackend). CompositeBackend mixes them.

### FilesystemBackend — writing to real disk

When the agent needs to work with **actual files**, use `FilesystemBackend` with `virtual_mode=True` to sandbox under `root_dir`.

> ⚠️ Always sandbox with `virtual_mode=True` to prevent path traversal.


In [5]:
from deepagents.backends import FilesystemBackend
import tempfile
import os
import shutil

sandbox_dir = tempfile.mkdtemp(prefix="deepagents_sandbox_")
fs_backend = FilesystemBackend(root_dir=sandbox_dir, virtual_mode=True)

agent_with_fs = create_deep_agent(
    model=model,
    backend=fs_backend,
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": str(uuid7())}}
agent_with_fs.invoke({
    "messages": [{
        "role": "user", 
        "content": "Write predictions.txt with your World Cup outcome predictions for the top 10 teams."}]
}, config=config)

# Verify it actually hit disk
actual_path = os.path.join(sandbox_dir, "predictions.txt")
with open(actual_path) as f:
    print(f"File on disk at {actual_path}: {f.read()}")

File on disk at /var/folders/lf/frbcgxcd5fj9kkk3b16cq9480000gn/T/deepagents_sandbox_bvn3fgve/predictions.txt: World Cup Outcome Predictions for Top 10 Teams

1. Argentina — Champions
2. France — Runners-up
3. Brazil — Semi-finals
4. England — Semi-finals
5. Spain — Quarter-finals
6. Portugal — Quarter-finals
7. Germany — Round of 16
8. Netherlands — Quarter-finals
9. Italy — Did not qualify / not participating
10. Belgium — Round of 16



In [6]:
# Remove the file
shutil.rmtree(sandbox_dir, ignore_errors=True)

## Part 4: Adding a Research Subagent

<img src="./images/deepAgentSubagents.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Subagent Architecture">

As agents do more work, their context fills with intermediate tool calls. **Subagents** isolate work in a separate context — only the final summary returns to the main agent.

```
Without subagents: every search result balloons the main agent's context
With subagents:    subagent searches in isolation, returns one clean summary
```

**Key takeaway:** delegate via `subagents=[...]`. The main agent calls the subagent through a `task()` tool and only sees the result, not the intermediate tool calls. By default `create_deep_agent` lets a subagent **inherit the parent's tools** — pass `tools=[...]` in the subagent spec only to override. (If you instantiate `SubAgentMiddleware` directly, `tools` is required.)


In [7]:
from utils.models import sub_agent_model
print(sub_agent_model)

output_version=None profile={'name': 'GPT-5.4 mini', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x10f610e10> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10f6111d0> root_client=<openai.OpenAI object at 0x10f610b90> root_async_client=<openai.AsyncOpenAI object at 0x10f610f50> model_name='gpt-5.4-mini' model_kwargs={} openai_api_key=SecretStr('**********') opena

In [8]:
research_subagent = {
    "name": "research-agent",
    "description": "Delegate focused research tasks.",
    "system_prompt": (
        "You are a research assistant. \n"
        "- Use 1-2 searches maximum, then summarize.\n"
        "- Cite with inline numbers [1], [2] and end with a Sources section."
    ),
    # No "tools" key → inherits the parent's tools (tavily_search).
    "model": sub_agent_model,
}

agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are a research coordinator. Always delegate research to subagents.",
    subagents=[research_subagent],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{
        "role": "user", 
        "content": "Research how the US Men's National Team could improve its performance" 
        "for the next World Cup, and give me a short summary of the top opportunities."}]
}, config=config)
print(result["messages"][-1].content)


Top opportunities for the USMNT before the next World Cup:

- **Improve midfield balance**: The team needs a more stable structure in possession and better protection against counterattacks. A clearer balance of ball progression, defensive coverage, and link play could make them harder to break.
- **Create more repeatable attacking patterns**: The attack too often depends on individual talent. Better coordination between wingers, fullbacks, and central runners would produce more reliable chances.
- **Define the striker role better**: The US has lacked consistent No. 9 production. Picking a clearer forward profile and building the attack around that role could improve finishing output.
- **Make set pieces a real weapon**: Tournament matches are often decided by small margins. Better attacking routines and defensive organization on dead balls could add goals without needing better open-play dominance.
- **Prioritize roster fit and depth**: The best squad is not just the most talented one

## Part 5: Human-in-the-Loop

<img src="./images/deepAgentHITL.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Human-in-the-Loop">

For sensitive operations, you may want a human to approve before the agent acts. Deep Agents has interrupt support built in via `HumanInTheLoopMiddleware`.

**Built-in decision types:** Approve · Edit · Reject.

You can scope per tool:

```python
interrupt_on = {
    "delete_file": {"allowed_decisions": ["approve", "edit", "reject"]},
    "write_file": {"allowed_decisions": ["approve", "reject"]},
}
```

**Key takeaway:** configure `interrupt_on={...}` to gate risky tools. A checkpointer is required (the agent must pause and resume).

> **Try it live:** the first cell runs the agent until it hits an interrupt. The second cell reviews it **interactively** — it calls `input()`, so the notebook pauses and waits for you to type `approve`, `edit`, or `reject` before the graph continues. Resuming is just another `invoke()` with a `Command(resume=...)` payload carrying your decision.

In [9]:
# Interrupt on file writes/edits
agent_with_hitl = create_deep_agent(
    model=model,
    tools=[tavily_search],
    checkpointer=checkpointer,
    interrupt_on={
        "write_file": {"allowed_decisions": ["approve", "edit", "reject"]},
        "edit_file": {"allowed_decisions": ["approve", "edit", "reject"]},
    },
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_hitl.invoke({
    "messages": [{"role": "user", "content": "Write a file called /predictions.md with '2. Morocco'"}]
}, config=config)

if result.get("__interrupt__"):
    print("Interrupt triggered!")
    iv = result["__interrupt__"][0].value
    for action, review in zip(iv["action_requests"], iv["review_configs"]):
        print(f"  Tool: {action['name']}  Args: {action['args']}")
        print(f"  Allowed: {review['allowed_decisions']}")


Interrupt triggered!
  Tool: write_file  Args: {'file_path': '/predictions.md', 'content': '2. Morocco'}
  Allowed: ['approve', 'edit', 'reject']


In [10]:
from utils.hitl import review_interrupts

# review_interrupts prompts for approve/edit/reject on each pending tool call
# and resumes the graph. input() blocks the cell until you respond.
result = review_interrupts(agent_with_hitl, result, config)

print("\nGraph finished:", result["messages"][-1].content)
files = result.get("files", {})
print("Files now in state:", list(files.keys()))
file_data = files.get("/predictions.md") or {}
print("File content:", file_data.get("content", "(not found)"))


Tool call awaiting review: write_file
   Args: {'file_path': '/predictions.md', 'content': '2. Morocco'}


   Decision ['approve', 'edit', 'reject'] (default: approve) >  approve



Graph finished: Done.
Files now in state: ['/predictions.md']
File content: 2. Morocco


## Part 6: Long-Term Memory

<img src="./images/deepAgentMemories.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Long-Term Memory">

Files in StateBackend disappear when a thread ends. **Long-term memory** uses `CompositeBackend` to route specific paths to a persistent `StoreBackend`.

```
/memories/*       →  StoreBackend  (persistent, cross-thread)
everything else   →  StateBackend  (ephemeral)
```

**Key takeaway:** mix backends with `CompositeBackend` and pass a `store=` for the persistent layer. In production, LangGraph Platform / LangSmith Deployments provides the store automatically.


In [15]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend

store = InMemoryStore()

# Compose: /memories/* persists in the store; everything else stays ephemeral
backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend()},
)

agent_with_memory = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt=(
        "You are a helpful assistant with long-term memory.\n"
        "Save important notes to /memories/ so they persist across conversations.\n"
        "Files outside /memories/ disappear when the conversation ends."
    ),
    checkpointer=checkpointer,
    backend=backend,
    store=store,
)

# Thread 1: save to memory
config1 = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_memory.invoke({
    "messages": [{"role": "user", "content": "Save 'It seemed like the US had a chance in the 2026 World Cup.' to `/memories/world_cup.md`"}]
}, config=config1)
print("Thread 1:", result["messages"][-1].content)

# Prove the routing: StateBackend stays empty; the store holds the file.
state_files = list(result.get("files", {}).keys())
print(f"\nStateBackend files (ephemeral, per-thread): {state_files}")

store_items = store.search(("filesystem",))
print("StoreBackend items (persistent, cross-thread):")
for item in store_items:
    content = item.value.get("content", "")
    print(f"  {item.key} -> {content!r}")

Thread 1: Saved to `/memories/world_cup.md`.

StateBackend files (ephemeral, per-thread): []
StoreBackend items (persistent, cross-thread):
  /world_cup.md -> 'It seemed like the US had a chance in the 2026 World Cup.\n'


In [16]:
# Thread 2: brand-new thread, but the memory persists!
config2 = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_memory.invoke({
    "messages": [{"role": "user", "content": "Read me what's in `/memories/world_cup.md`"}]
}, config=config2)
print("Thread 2:", result["messages"][-1].content)


Thread 2: It seemed like the US had a chance in the 2026 World Cup.


## Part 7: AGENTS.md & Skills

So far we've put instructions in `system_prompt`. Two file-based alternatives are more powerful:

| Approach | Loaded when | Editable by agent | Best for |
|---|---|---|---|
| `system_prompt` | Always | No | Core identity, immutable rules |
| `AGENTS.md` (`memory=`) | Always | **Yes** | Workflow, learnable rules |
| `SKILL.md` (`skills=`) | On demand | No | Task-specific templates |

- **`AGENTS.md`** is loaded into the system prompt via `memory=`; the agent can read AND edit its own `AGENTS.md`.
- **Skills** use **progressive disclosure**: only the skill's name + description loads at startup; the full `SKILL.md` is read on demand when the agent decides it's relevant.

> **Path convention:** in this notebook we use absolute virtual paths (`/AGENTS.md`, `/skills/`) because the default `StateBackend` treats `/` as the root of the virtual filesystem, and we seed those files via `files=` in the next cell. The standalone agent at `agent/agent.py` uses relative paths (`./AGENTS.md`, `./skills/`) because it points a real `FilesystemBackend` at `agent/` via `root_dir=`, where those files already live on disk.



In [17]:
# AGENTS.md — the agent's identity + workflow (always loaded)
agents_md = """# Research Assistant

You search the web, synthesize findings, and produce polished content.

## Workflow
1. **Plan** — use `write_todos` to break down the task
2. **Research** — search via tavily_search (1-2 searches max)
3. **Synthesize** — combine findings into a brief report
4. **Remember** — save key takeaways to `/memories/research_notes.md`

## Rules
- Cite with inline numbers [1], [2]; consolidate duplicate URLs
- End reports with a Sources section
- Check loaded skills before formatting specialized content
"""

# A skill — loaded on demand when the agent sees a LinkedIn-shaped task
linkedin_skill = """---
name: linkedin-post
description: Write a LinkedIn post from research findings. Use for professional posts or thought-leadership.
---

# LinkedIn Post Skill

## Format
- Hook line that grabs attention before the 'see more' cut
- 3-5 short paragraphs (1-2 sentences each), with line breaks between
- 1-2 emojis per paragraph (don't overdo)
- End with a question or CTA + 3-5 hashtags

## Length
- 150-300 words. First line must hook the reader.
"""

In [20]:
from deepagents.backends.utils import create_file_data

skill_agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    checkpointer=checkpointer,
    backend=backend,
    store=store,
)

# Seed the virtual filesystem so memory= and skills= can find the files
files = {
    "/AGENTS.md": create_file_data(agents_md),
    "/skills/linkedin-post/SKILL.md": create_file_data(linkedin_skill),
}

config = {"configurable": {"thread_id": str(uuid7())}}
result = skill_agent.invoke({
    "messages": [{
        "role": "user", 
        "content": "Briefly recap Morocco's 2026 World Cup journey and write a short LinkedIn post about "
        "how their journey relates to B2B sales. The post should be fit for getting featured on the LinkedInLunatics subreddit."
    }],
    "files": files,
}, config=config)
rprint(result["messages"][-1].content)

Morocco’s 2026 World Cup story, briefly: they qualified strongly, became the first African side to book a spot at 
the tournament, then carried that momentum into the finals with an unbeaten group stage and a Round of 16 win over 
Canada before falling to France in the quarterfinals. In short: fast start, disciplined execution, real belief, and
then the brutal reminder that good runs still end if you can’t clear the very top tier.

**LinkedIn post:**

Morocco’s 2026 World Cup run is basically B2B sales in shin guards ⚽📈

They qualified early, stayed unbeaten in the group, crushed Canada in the Round of 16, and then got stopped by 
France in the quarterfinals.

Here’s the sales lesson: momentum gets you meetings. Discipline gets you pipeline. But elite execution is what 
closes at the top end. 🎯💼

Too many teams celebrate “strong activity” the way fans celebrate possession stats. More calls. More demos. More 
outbound. More “energy.”  
But Morocco’s run is a reminder that progress comes from structure, resilience, and making key moments count. 🔥📊

In B2B sales, your group stage is prospecting.  
Your knockout round is discovery and stakeholder management.  
And your quarterfinal? That’s procurement, legal, and a CFO who suddenly has “one quick question.” 😅🤝

The teams that win aren’t always the flashiest. They’re the ones that know their system, trust it, and execute 
under pressure.

Are you building a sales team that can win qualifiers… or one that can go deep in the tournament? 👀  
#B2BSales #SalesLeadership #Pipeline #RevenueOperations #WorldCup

If you want, I can make it even more unhinged so it feels *perfectly* subreddit-featured.

## Part 8: Wrap-Up & Next Steps

Starting from a basic `create_deep_agent()`, we progressively added:

```
Part 1: create_deep_agent(model)              → Filesystem + planning
Part 2: + tools=[tavily_search]               → Web search
Part 3: (backends)                            → Storage model: state vs disk
Part 4: + subagents=[research_subagent]       → Context isolation
Part 5: + interrupt_on={...}                  → Human oversight
Part 6: + backend=CompositeBackend + store    → Long-term memory
Part 7: + memory=AGENTS.md + skills=SKILL.md  → File-based identity + on-demand capabilities
```

### Resources
- [Deep Agents docs](https://docs.langchain.com/oss/python/deepagents/)
- [LangChain Academy](https://academy.langchain.com/)
- [LangChain vs LangGraph vs Deep Agents](https://docs.langchain.com/oss/python/concepts/products)

### What to try next
1. **Run in Studio** — wrap your agent for `langgraph dev` or LangSmith Deployments
2. **More skills** — write SKILL.md files for your domain
3. **Per-user memory** — namespace `StoreBackend` by `user_id`
4. **Multi-agent systems** — compose multiple specialized subagents (try using a cheaper open model for subagents!)

---

**Happy building!** 🚀
